### Aim: a station name crosswalk between chuuchuu and SNCF's open data

nb5's SNCF-comparable export (`chuuchuu_summary_sncf_format_hsr.xlsx`) uses chuuchuu's own station names (e.g. "Paris-Gare-de-Lyon"), while SNCF's own open dataset (`sup_data/regularite-mensuelle-tgv-sncf.csv`) uses a different, coarser naming convention (e.g. "PARIS LYON"). Comparing the two directly needs a name-to-name crosswalk -- that's what this notebook builds.

Approach: normalize both station-name universes (strip accents, uppercase, collapse punctuation, expand a few known abbreviations) and score every chuuchuu station against every SNCF station by token overlap. High-confidence matches are accepted automatically; everything else is reviewed by hand, because token overlap alone can't tell "Avignon Centre" (an in-city station) from "Avignon TGV" (a different station 4km away), or recognise that "Francfort sur le Main" IS "FRANCFORT" despite extra words diluting the score.

The result is saved as a reusable reference table: `sup_data/station_name_mapping_chuuchuu_sncf.csv`.

In [ ]:
import pandas as pd
import unicodedata
import re
import os

sncf_df = pd.read_csv("sup_data/regularite-mensuelle-tgv-sncf.csv", sep=";", encoding="utf-8-sig")
sncf_stations = sorted(set(sncf_df["Gare de départ"].dropna()) | set(sncf_df["Gare d'arrivée"].dropna()))

# same HSR scope and depart/terminus attribution as nb5's SNCF-format export
data_chuuchuu = pd.read_parquet(
    "intermediate_outputs/data_chuuchuu_french_cancellations.parquet",
    columns=["routeType", "depart_terminus", "stopName", "stopName_slug", "country",
             "journey_id", "is_ambiguous_trip", "cross_agency_duplicate_confidence"],
)
clean = ~data_chuuchuu["is_ambiguous_trip"] & (data_chuuchuu["cross_agency_duplicate_confidence"] != "likely_true_duplicate")
HSR_ROUTE_TYPES = ["ICE", "LYR", "LYRIA", "TGV INOUI", "OUI", "FR", "EST"]
hsr_ends = data_chuuchuu[
    clean
    & data_chuuchuu["routeType"].isin(HSR_ROUTE_TYPES)
    & data_chuuchuu["depart_terminus"].isin(["depart", "terminus"])
]
chuuchuu_stations = hsr_ends[["stopName", "stopName_slug", "country"]].drop_duplicates().dropna(subset=["stopName"])

print(f"{len(sncf_stations)} distinct SNCF station labels, {len(chuuchuu_stations)} distinct chuuchuu HSR origin/destination stations")

### Automatic first pass: normalized token overlap

Two names are compared as sets of normalized tokens (accents stripped, uppercased, punctuation collapsed, "ST"/"STE" expanded to "SAINT"/"SAINTE", a couple of generic words like "GARE"/"HALL"/"VOYAGEURS" dropped as noise). The score is the Dice coefficient (2 x shared tokens / total tokens), which tolerates one side having extra descriptive words -- e.g. "Paris Gare de Lyon Hall 1 - 2" vs "PARIS LYON".

In [ ]:
SYNONYMS = {"ST": "SAINT", "STE": "SAINTE", "BARCELONE": "BARCELONA", "ZUERICH": "ZURICH"}
NOISE_TOKENS = {"HALL", "1", "2", "GARE", "VOYAGEURS", "INT", "L"}


def normalize_tokens(name):
    s = unicodedata.normalize("NFKD", str(name))
    s = "".join(c for c in s if not unicodedata.combining(c)).upper()
    s = re.sub(r"[^A-Z0-9]+", " ", s)
    tokens = [SYNONYMS.get(t, t) for t in s.split() if t]
    return {t for t in tokens if t not in NOISE_TOKENS}


def dice_score(a, b):
    if not a or not b:
        return 0.0
    return 2 * len(a & b) / (len(a) + len(b))


sncf_tokens = {s: normalize_tokens(s) for s in sncf_stations}


def best_candidate(name):
    toks = normalize_tokens(name)
    scored = sorted(((dice_score(toks, t), s) for s, t in sncf_tokens.items()), reverse=True)
    return scored[0] if scored else (0.0, None)


candidates = chuuchuu_stations.copy()
candidates[["score", "candidate"]] = candidates["stopName"].apply(lambda n: pd.Series(best_candidate(n)))

AUTO_ACCEPT_THRESHOLD = 0.75
print(f"{(candidates['score'] >= AUTO_ACCEPT_THRESHOLD).sum()} stations auto-accepted (score >= {AUTO_ACCEPT_THRESHOLD})")
print(f"{(candidates['score'] < AUTO_ACCEPT_THRESHOLD).sum()} stations need manual review")
candidates.sort_values("score", ascending=False).head(10)

### Manual review

Every station below the auto-accept threshold was checked by hand against a map/timetable. They fall into a few buckets:

- **Confirmed correct despite a mid-range score** -- e.g. "Barcelone-Sants" / "BARCELONA" (spelling varies by language), "Francfort sur le Main" / "FRANCFORT" (extra words diluted the score), "Lille Europe" and "Lille Flandres" both map to SNCF's single city-level "LILLE" label.
- **Reassigned** -- "Paris Montparnasse Vaugirard" auto-matched to "PARIS MONTPARNASSE", but Vaugirard is a distinct annex station; chuuchuu already has a separate "Paris Montparnasse Hall 1 - 2" entry for the main hall, so Vaugirard belongs to SNCF's separate "PARIS VAUGIRARD" label instead.
- **Confirmed NOT a match** -- same city, different physical station, and SNCF's file doesn't track the chuuchuu one separately: Avignon Centre vs Avignon TGV, Montpellier Sud de France vs Montpellier Saint-Roch, Nîmes Pont du Gard vs Nîmes Centre, Lyon Perrache vs Lyon Part-Dieu, Paris Austerlitz, Valence Ville vs Valence TGV. These stay unmapped rather than being forced onto a same-city label that means something different.
- **Country-level aggregate** -- SNCF's file reports every Italian destination under a single "ITALIE" label (no per-station breakdown), so all 4 chuuchuu Italian stations (Milano Centrale/Certosa/Porta Garibaldi, Oulx) map to it.
- **Out of scope** -- most remaining unmatched stations (Belgium, Netherlands, Luxembourg, UK, and most German cities beyond Frankfurt/Stuttgart) simply aren't in this SNCF file at all: it only covers SNCF/Lyria-branded TGV routes, not Eurostar/Thalys/Izy, and doesn't track German ICE destinations past Frankfurt and Stuttgart.

In [ ]:
# each entry: chuuchuu stopName_slug -> (sncf_station_label_or_None, note)
MANUAL_OVERRIDES = {
    # confirmed correct despite a mid-range automatic score
    "barcelone-sants": ("BARCELONA", "spelling variant (FR Barcelone / SNCF Barcelona)"),
    "bellegarde": ("BELLEGARDE (AIN)", "same station, SNCF appends the department"),
    "dijon": ("DIJON VILLE", "same station"),
    "disneyland-paris-marne-la-vallee-chessy": ("MARNE LA VALLEE", "same station"),
    "geneve-cornavin": ("GENEVE", "Cornavin is Geneva's main station"),
    "lille-europe": ("LILLE", "SNCF reports Lille as a single city-level label"),
    "lille-flandres": ("LILLE", "SNCF reports Lille as a single city-level label"),
    "mulhouse": ("MULHOUSE VILLE", "same station"),
    "nimes-centre": ("NIMES", "same station"),
    "stuttgart-hbf": ("STUTTGART", "same station"),
    "zuerich-hb": ("ZURICH", "spelling variant (DE Zuerich / FR Zurich)"),
    "montpellier-saint-roch": ("MONTPELLIER", "same station (the historic city-centre one)"),
    "valence-tgv-rhone-alpes-sud": ("VALENCE ALIXAN TGV", "same station, fuller chuuchuu name"),
    "francfort-sur-le-main": ("FRANCFORT", "same city; extra words diluted the auto score"),
    # reassigned: automatic pick was a same-city but wrong station
    "paris-montparnasse-vaugirard": ("PARIS VAUGIRARD", "Vaugirard is a distinct annex to Gare Montparnasse -- "
                                                          "chuuchuu already has 'Paris Montparnasse Hall 1 - 2' for the main hall"),
    # confirmed NOT a match: different, physically distinct station SNCF doesn't track here
    "avignon-centre": (None, "different station from Avignon TGV (~4km apart), not in the SNCF file"),
    "calais-ville": (None, "not in the SNCF file; auto-matched only on the generic word VILLE"),
    "le-croisic": (None, "not in the SNCF file; auto-matched only on the generic word LE"),
    "le-havre": (None, "not in the SNCF file; auto-matched only on the generic word LE"),
    "lorraine-tgv": (None, "not in the SNCF file; auto-matched only on the generic word TGV"),
    "lyon-perrache": (None, "different station from Lyon Part-Dieu, not in the SNCF file"),
    "lyon-perrache-voyageurs": (None, "different station from Lyon Part-Dieu, not in the SNCF file"),
    "paris-austerlitz": (None, "different Paris terminus, not in the SNCF file"),
    "saint-brieuc": (None, "not in the SNCF file; auto-matched only on SAINT"),
    "saint-die-des-vosges": (None, "not in the SNCF file; auto-matched only on SAINT/DES"),
    "saint-jean-de-luz-ciboure": (None, "not in the SNCF file; auto-matched only on SAINT/JEAN"),
    "saint-jean-de-maurienne-arvan": (None, "not in the SNCF file; auto-matched only on SAINT/JEAN"),
    "saint-nazaire": (None, "not in the SNCF file; auto-matched only on SAINT"),
    "valence-ville": (None, "different station from Valence TGV, not in the SNCF file"),
    "montpellier-sud-de-france": (None, "different, newer TGV station from Montpellier Saint-Roch, not in the SNCF file"),
    "nimes-pont-du-gard": (None, "different, newer TGV station from Nimes Centre, not in the SNCF file"),
    # country-level aggregate: SNCF reports all Italian destinations under one label
    "milano-porta-garibaldi": ("ITALIE", "SNCF reports all Italian destinations as a single country-level label"),
    "milano-centrale": ("ITALIE", "SNCF reports all Italian destinations as a single country-level label"),
    "milano-certosa": ("ITALIE", "SNCF reports all Italian destinations as a single country-level label"),
    "oulx-cesana-claviere-sestriere": ("ITALIE", "SNCF reports all Italian destinations as a single country-level label"),
}


def resolve(row):
    slug = row["stopName_slug"]
    if slug in MANUAL_OVERRIDES:
        label, note = MANUAL_OVERRIDES[slug]
        return pd.Series({"sncf_station_label": label, "match_source": "manual", "match_note": note})
    if row["score"] >= AUTO_ACCEPT_THRESHOLD:
        return pd.Series({"sncf_station_label": row["candidate"], "match_source": "auto", "match_note": ""})
    return pd.Series({"sncf_station_label": None, "match_source": "unmatched",
                       "match_note": "no SNCF equivalent found (outside SNCF's TGV/Lyria reporting scope, "
                                     "or a country/brand this file doesn't cover)"})


resolved = candidates.join(candidates.apply(resolve, axis=1))
station_mapping = resolved[["stopName", "stopName_slug", "country", "sncf_station_label", "match_source", "match_note", "score"]]
station_mapping = station_mapping.rename(columns={"score": "auto_match_score"}).sort_values(["country", "stopName"])

print(station_mapping["match_source"].value_counts())
print(f"{station_mapping['sncf_station_label'].notna().sum()} of {len(station_mapping)} stations mapped to an SNCF label")
station_mapping.head(15)

### Saving the crosswalk as a reusable reference table

In [ ]:
os.makedirs("sup_data", exist_ok=True)
mapping_path = "sup_data/station_name_mapping_chuuchuu_sncf.csv"
station_mapping.to_csv(mapping_path, index=False, encoding="utf-8-sig")
print(f"Saved to {mapping_path}")

### Validating the crosswalk: joining nb5's SNCF-format export against the real SNCF file

This isn't a new deliverable -- it's a sanity check that the crosswalk actually lets the two datasets be joined. It reuses `summary_stats/chuuchuu_summary_sncf_format_hsr.xlsx` from nb5 (run that notebook's export cell first if the file is missing).

**Important caveat this join surfaces:** nb5's `Gare d'arrivée` is the journey's true final destination (its `terminus`), while SNCF's file reports figures for a *liaison* (a commercial city-pair) regardless of whether the train continues past it. A Paris-Marseille TGV that merely passes through Lyon never counts as "Paris-Lyon" in nb5's table, but it does count toward SNCF's "PARIS LYON -> LYON PART DIEU" liaison. That's why, below, chuuchuu's train count for that pair comes out far lower than SNCF's for the same month -- the station names are correctly matched, but the two tables are counting different things. Building a true liaison-level (intermediate-stop) version of nb5's export -- checking delay/cancellation at *any* matching intermediate stop, not just the final terminus -- would be a separate follow-up if an exact volume comparison is needed.

In [ ]:
name_to_sncf = station_mapping.set_index("stopName")["sncf_station_label"].to_dict()

chuuchuu_hsr = pd.read_excel("summary_stats/chuuchuu_summary_sncf_format_hsr.xlsx")
chuuchuu_hsr["sncf_gare_depart"] = chuuchuu_hsr["Gare de départ"].map(name_to_sncf)
chuuchuu_hsr["sncf_gare_arrivee"] = chuuchuu_hsr["Gare d'arrivée"].map(name_to_sncf)
chuuchuu_hsr["sncf_service"] = chuuchuu_hsr["Service"].map({"domestic": "National", "international": "International"})

mapped = chuuchuu_hsr.dropna(subset=["sncf_gare_depart", "sncf_gare_arrivee"])
print(f"{len(mapped)} of {len(chuuchuu_hsr)} chuuchuu Date/Service/OD/Route-type rows have both stations mapped to an SNCF label")

# collapse chuuchuu's route-type split back to SNCF's grain: one row per Date/Service/OD
chuuchuu_collapsed = mapped.groupby(["Date", "sncf_service", "sncf_gare_depart", "sncf_gare_arrivee"]).agg(
    chuuchuu_n_total_train=("Nombre de circulations prévues", "sum"),
    chuuchuu_n_cancelled=("Nombre de trains annulés", "sum"),
).reset_index()

sncf_slim = sncf_df[["Date", "Service", "Gare de départ", "Gare d'arrivée",
                      "Nombre de circulations prévues", "Nombre de trains annulés"]].rename(columns={
    "Nombre de circulations prévues": "sncf_n_total_train",
    "Nombre de trains annulés": "sncf_n_cancelled",
})

comparison = chuuchuu_collapsed.merge(
    sncf_slim,
    left_on=["Date", "sncf_service", "sncf_gare_depart", "sncf_gare_arrivee"],
    right_on=["Date", "Service", "Gare de départ", "Gare d'arrivée"],
    how="inner",
)
print(f"{len(comparison)} Date/Service/OD rows found in both datasets")

# example: Paris -> Lyon Part-Dieu, June 2025
comparison[
    (comparison["sncf_gare_depart"] == "PARIS LYON")
    & (comparison["sncf_gare_arrivee"] == "LYON PART DIEU")
    & (comparison["Date"] == "2025-06")
][["Date", "sncf_service", "sncf_gare_depart", "sncf_gare_arrivee",
   "chuuchuu_n_total_train", "sncf_n_total_train", "chuuchuu_n_cancelled", "sncf_n_cancelled"]]